<a href="https://colab.research.google.com/github/mathewseriese-bit/flutter-ui-and-animations/blob/main/notebooks/basic_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [106]:
!pip install microwakeword mmap-ninja

# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [105]:
# Re-installing microWakeWord and forcing path inclusion
import os
import sys
import platform

if platform.system() == "Darwin":
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

if not os.path.exists('microWakeWord'):
    !git clone https://github.com/kahrendt/microWakeWord

!pip install -e ./microWakeWord

# Add the package to the Python path manually to avoid import errors
mww_path = os.path.abspath('./microWakeWord')
if mww_path not in sys.path:
    sys.path.append(mww_path)

print("microWakeWord path added to sys.path")

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-amukcz5y
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-amukcz5y
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Obtaining file:///content/microWakeWord
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ...

In [9]:
target_word = 'hey_koda'  # Phonetic spellings may produce better samples

import os
import sys
import platform

from IPython.display import Audio

repo_path = "./piper-sample-generator"
generate_script_relative_path = "generate_samples.py"
generate_script_full_path = os.path.join(repo_path, generate_script_relative_path)
output_dir = "generated_samples"
audio_file_path = os.path.join(output_dir, "0.wav")

# Check if the repository directory exists, and clone if it doesn't.
if not os.path.exists(repo_path):
    print(f"Cloning {repo_path}...")
    if platform.system() == "Darwin":
        # Note: Added {repo_path} to ensure cloning into the specified directory
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator {repo_path}
    else:
        # Note: Added {repo_path} to ensure cloning into the specified directory
        !git clone https://github.com/rhasspy/piper-sample-generator {repo_path}

    # Download the pre-trained model for Piper TTS
    print("Downloading Piper TTS model...")
    !wget -O {repo_path}/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

    # Install Python dependencies required by piper-sample-generator
    print("Installing Python dependencies: torch, torchaudio, piper-phonemize-cross...")
    !pip install torch torchaudio piper-phonemize-cross==1.2.1

    # Add the cloned repository to Python's system path for module imports, if needed by other parts of the notebook
    if repo_path not in sys.path:
        sys.path.append(repo_path)
else:
    print(f"Repository {repo_path} already exists. Skipping clone and download.")

# --- Error Diagnosis and Fix for "No such file or directory" ---
# The original error indicates that 'generate_samples.py' could not be found.
# This check verifies its existence after cloning or if the repo already existed.
if not os.path.exists(generate_script_full_path):
    print(f"ERROR: The script '{generate_script_full_path}' was not found. Please ensure the git clone was successful and the file exists.")
    # If the file is genuinely missing, we cannot proceed. Raise an error.
    raise FileNotFoundError(f"Required script '{generate_script_full_path}' not found.")
else:
    print(f"Successfully located script: {generate_script_full_path}")

# Ensure the output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Execute the sample generation script
# Using the full absolute path for the script ensures there are no ambiguities with the current working directory.
print(f"Generating sample for '{target_word}'...")
!python3 "{generate_script_full_path}" \
--target-word "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir "{output_dir}"

# Check if the audio file was generated before attempting to play it
if os.path.exists(audio_file_path):
    print(f"Playing generated audio: {audio_file_path}")
    Audio(audio_file_path, autoplay=True)
else:
    print(f"WARNING: Audio file '{audio_file_path}' was not found. Sample generation might have failed.")

Repository ./piper-sample-generator already exists. Skipping clone and download.
Successfully located script: ./piper-sample-generator/generate_samples.py
Generating sample for 'hey_koda'...
Traceback (most recent call last):
  File "/content/./piper-sample-generator/generate_samples.py", line 20, in <module>
    from espeak_phonemizer import Phonemizer
ModuleNotFoundError: No module named 'espeak_phonemizer'


In [10]:
!rm -rf /content/piper-sample-generator
!git clone https://github.com/dscripka/piper-sample-generator
!mkdir -p /content/piper-sample-generator/models
!wget -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 75 (delta 21), reused 18 (delta 18), pack-reused 40 (from 1)
Receiving objects: 100% (75/75), 1.01 MiB | 3.20 MiB/s, done.
Resolving deltas: 100% (22/22), done.
--2026-04-30 10:46:02--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-30T11%3A29%3A03Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-

In [11]:
!pip install webrtcvad

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73518 sha256=6c8a1f296407130e4438b6f96cde9706ac11b251a6e86d31f59178354343fbd3
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad


In [12]:
import os
from IPython.display import Audio

target_word = 'hey_koda'
output_dir = 'quick_sample'
if not os.path.exists(output_dir): os.makedirs(output_dir)

# Generate 1 sample
!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir "{output_dir}"

# Play it
sample_path = os.path.join(output_dir, '0.wav')
if os.path.exists(sample_path):
    display(Audio(sample_path, autoplay=True))
else:
    print('Failed to generate sample. Please check if the model exists in piper-sample-generator/models/')

Traceback (most recent call last):
  File "/content/piper-sample-generator/generate_samples.py", line 20, in <module>
    from espeak_phonemizer import Phonemizer
ModuleNotFoundError: No module named 'espeak_phonemizer'
Failed to generate sample. Please check if the model exists in piper-sample-generator/models/


In [14]:
!pip install espeak-phonemizer

In [15]:
import os

samples_dir = 'generated_samples'
if os.path.exists(samples_dir):
    wav_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]
    print(f"Found {len(wav_files)} .wav files in '{samples_dir}'.")
    if wav_files:
        print("First 5 files:", wav_files[:5])
else:
    print(f"Directory '{samples_dir}' does not exist.")

Found 0 .wav files in 'generated_samples'.


In [27]:
# Install system dependencies and run generation
import os

# Install libespeak-ng which is required by the phonemizer
!apt-get update && apt-get install -y libespeak-ng1

# Ensure the config file exists
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json'

# Patch script for PyTorch 2.6 and weights loading
script_path = 'piper-sample-generator/generate_samples.py'
with open(script_path, 'r') as f:
    lines = f.readlines()

patch = """
import torch
try:
    from piper_train.vits.models import SynthesizerTrn, TextEncoder
    torch.serialization.add_safe_globals([SynthesizerTrn, TextEncoder])
except ImportError:
    pass
"""

new_content = "".join(lines)
if 'torch.load(model_path)' in new_content:
    new_content = new_content.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)')
if 'torch.serialization.add_safe_globals' not in new_content:
    new_content = patch + new_content

with open(script_path, 'w') as f:
    f.write(new_content)

# Run the generation with the new phonetic spelling
target_word = 'hey koh dah'
model_path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--model "{model_path}" \
--max-samples 1000 \
--batch-size 100 \
--output-dir generated_samples

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [51]:
import os
import random
from IPython.display import Audio

# List all generated wav files
samples_dir = 'generated_samples'
sample_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]

if sample_files:
    # Pick a random sample and play it
    random_sample = random.choice(sample_files)
    sample_path = os.path.join(samples_dir, random_sample)
    print(f'Playing: {sample_path}')
    display(Audio(sample_path, autoplay=True))
else:
    print('No samples found in the directory. Please run the generation cell first.')

Playing: generated_samples/351.wav


In [102]:
import os
import random
from IPython.display import Audio, display

# List all saved RIR files
rir_dir = './mit_rirs'
if os.path.exists(rir_dir):
    rir_files = [f for f in os.listdir(rir_dir) if f.endswith('.wav')]

    if rir_files:
        # Pick a random RIR and play it
        random_rir = random.choice(rir_files)
        rir_path = os.path.join(rir_dir, random_rir)
        print(f'Playing RIR sample: {rir_path}')
        display(Audio(rir_path, autoplay=True))
    else:
        print(f'No .wav files found in {rir_dir}. Please ensure the download cell finished.')
else:
    print(f'Directory {rir_dir} does not exist.')

Playing RIR sample: ./mit_rirs/rir_105.wav


### Step 3: Audio Augmentation and Feature Generation
We will now configure the `microwakeword` augmenter. This uses the RIRs and background noises we downloaded to create a diverse training set from your TTS samples.

In [107]:
import os
import sys

# Ensure the local microWakeWord path is included
mww_path = os.path.abspath('./microWakeWord')
if mww_path not in sys.path:
    sys.path.append(mww_path)

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# Define the paths for our datasets
impulse_paths = ['mit_rirs']
background_paths = ['fma_16k', 'audioset_16k']

# Configure the clips (our generated wake word samples)
clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=1.5,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

# Define the augmentation strategy
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities = {
        "SevenBandParametricEQ": 0.15,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.8,
        "Gain": 1.0,
        "RIR": 0.6,
    },
    impulse_paths=impulse_paths,
    background_paths=background_paths,
    background_min_snr_db=-5,
    background_max_snr_db=15,
    min_jitter_s=0.1,
    max_jitter_s=0.3,
)

print("Augmentation and Clips initialized successfully.")

Augmentation and Clips initialized successfully.


#### Generate and Save Spectrogram Features
This process converts the augmented audio into the format the neural network expects. It will save the features as memory-mapped files for efficient training.

In [108]:
import os
from mmap_ninja.ragged import RaggedMmap

output_features_dir = 'generated_augmented_features'
os.makedirs(output_features_dir, exist_ok=True)

splits = ["training", "validation", "testing"]

for split in splits:
    out_dir = os.path.join(output_features_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    split_name = "train" if split == "training" else ("validation" if split == "validation" else "test")
    repetition = 2 if split == "training" else 1
    slide_frames = 10 if split != "testing" else 1

    print(f"Generating features for {split} split...")

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide_frames,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

print("Feature generation complete.")

Generating features for training split...


0it [00:00, ?it/s]

Generating features for validation split...


0it [00:00, ?it/s]

Generating features for testing split...


0it [00:00, ?it/s]

Feature generation complete.


import yaml
import os

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_koda"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    }
]

config["training_steps"] = [5000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [10]
config["learning_rates"] = [0.001]
config["batch_size"] = 128
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["maximization_metric"] = "average_viable_recall"
config["minimization_metric"] = None
config["target_minimization"] = 0.9

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

print("Training configuration updated and saved to training_parameters.yaml")

In [116]:
import yaml
import os

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_koda"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    }
]

config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [10]
config["learning_rates"] = [0.001]
config["batch_size"] = 128
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["maximization_metric"] = "average_viable_recall"

# Fix: Added missing metrics required by the training script
config["minimization_metric"] = None
config["target_minimization"] = 0.9

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

print("Training configuration updated with minimization_metric and saved.")

Training configuration updated with minimization_metric and saved.


In [110]:
# Start the training process
!python3 -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
2026-04-30 11:29:24.459706: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777548564.461240   20039 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 204, 1, 

In [111]:
# Patching microwakeword/train.py to fix the .numpy() error
import os

train_file = 'microWakeWord/microwakeword/train.py'
with open(train_file, 'r') as f:
    content = f.read()

# Replace the problematic .numpy() calls with a safe check
if 'test_set_fp = result["fp"].numpy()' in content:
    content = content.replace('test_set_fp = result["fp"].numpy()', 'test_set_fp = result["fp"].numpy() if hasattr(result["fp"], "numpy") else result["fp"]')
    content = content.replace('test_set_fn = result["fn"].numpy()', 'test_set_fn = result["fn"].numpy() if hasattr(result["fn"], "numpy") else result["fn"]')

with open(train_file, 'w') as f:
    f.write(content)

print("Patch applied successfully. Restarting training...")

Patch applied successfully. Restarting training...


In [113]:
# Remove existing model directory to allow a clean restart and then run training
import shutil
import os

model_dir = 'trained_models/hey_koda'
if os.path.exists(model_dir):
    shutil.rmtree(model_dir)
    print(f"Removed existing directory: {model_dir}")

!python3 -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

Removed existing directory: trained_models/hey_koda
INFO:absl:Loading and analyzing data sets.
2026-04-30 11:31:32.063413: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777548692.064878   20708 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────

In [115]:
import shutil
import os

# Ensure we start fresh and avoid the previous KeyError or missing weights issues
model_dir = 'trained_models/hey_koda'
if os.path.exists(model_dir):
    shutil.rmtree(model_dir)
    print(f'Cleaned up {model_dir} for a fresh training run.')

# Run training with the updated parameters
!python3 -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

Cleaned up trained_models/hey_koda for a fresh training run.
INFO:absl:Loading and analyzing data sets.
2026-04-30 11:33:59.679796: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777548839.681283   21465 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼───────────

### Step 5: Export to TFLite
After training completes, run the following cell to convert the best model weights into a quantized `.tflite` file.

In [114]:
# Export the best weights to quantized TFLite
!python3 -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

print("Export complete. Check the 'trained_models/hey_koda' directory for the .tflite file.")

INFO:absl:Loading and analyzing data sets.
2026-04-30 11:33:01.019904: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777548781.021407   21171 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/microWakeWord/microwakeword/model_train_eval.py", line 424, in <module>
    model.load_weights(
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 555, in __init__
    fid = make_

In [89]:
import datasets
import scipy.io.wavfile
import os
import numpy as np
from tqdm import tqdm

# 1. Download/Load MIT RIR data
rir_dir = './mit_rirs'
os.makedirs(rir_dir, exist_ok=True)
print('Processing MIT Environmental Impulse Responses...')

# Loading dataset
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')

for i, row in enumerate(tqdm(rir_dataset, desc='Saving RIRs')):
    audio_data = row['audio']

    # Handle AudioDecoder or dictionary formats
    if isinstance(audio_data, dict):
        array = audio_data['array']
        path = audio_data.get('path', f'rir_{i}.wav')
    else:
        # Access attributes for AudioDecoder objects
        array = np.array(audio_data.array if hasattr(audio_data, 'array') else audio_data['array'])
        path = getattr(audio_data, 'path', f'rir_{i}.wav')

    name = os.path.basename(path)
    if not name.endswith('.wav'):
        name += '.wav'

    # Convert to 16-bit PCM and save
    scipy.io.wavfile.write(os.path.join(rir_dir, name), 16000, (array * 32767).astype(np.int16))

print(f'Done! Found {len(os.listdir(rir_dir))} files in {rir_dir}')

Processing MIT Environmental Impulse Responses...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs: 100%|██████████| 270/270 [00:00<00:00, 864.18it/s]

Done! Found 270 files in ./mit_rirs


In [87]:
import os

rir_dir = './mit_rirs'
if os.path.exists(rir_dir):
    files = sorted([f for f in os.listdir(rir_dir) if f.endswith('.wav')])
    print(f"Found {len(files)} .wav files in '{rir_dir}'.")
    if files:
        print("First 10 files:")
        for f in files[:10]:
            print(f" - {f}")
else:
    print(f"Directory '{rir_dir}' does not exist.")

Found 0 .wav files in './mit_rirs'.


In [85]:
import os

dirs_to_verify = ['mit_rirs', 'audioset_16k', 'fma_16k']

for d in dirs_to_verify:
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if f.endswith('.wav')]
        print(f'Directory: {d}')
        print(f'  Count: {len(files)} .wav files')
        if files:
            print(f'  Samples: {files[:3]}')
    else:
        print(f'Directory: {d} DOES NOT EXIST')
    print('-' * 20)

Directory: mit_rirs
  Count: 0 .wav files
--------------------
Directory: audioset_16k
  Count: 500 .wav files
  Samples: ['audioset_484.wav', 'audioset_75.wav', 'audioset_368.wav']
--------------------
Directory: fma_16k
  Count: 210 .wav files
  Samples: ['fma_195.wav', 'fma_194.wav', 'fma_55.wav']
--------------------


In [84]:
import datasets
import scipy.io.wavfile
import os
import numpy as np
from tqdm import tqdm

# 1. Download MIT RIR data
rir_dir = './mit_rirs'
os.makedirs(rir_dir, exist_ok=True)
print('Downloading MIT Environmental Impulse Responses...')

# Loading dataset
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')

for row in tqdm(rir_dataset, desc='Saving RIRs'):
    audio_data = row['audio']

    # Check if audio_data is a dictionary (Standard) or a torchcodec object
    if isinstance(audio_data, dict):
        name = os.path.basename(audio_data['path'])
        array = audio_data['array']
    else:
        # Attribute-based access for torchcodec.decoders.AudioDecoder
        name = os.path.basename(audio_data.path)
        array = audio_data.array

    scipy.io.wavfile.write(os.path.join(rir_dir, name), 16000, (array * 32767).astype(np.int16))

print(f'Done! Saved {len(os.listdir(rir_dir))} RIR files.')

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs:   0%|          | 0/270 [00:00<?, ?it/s]


AttributeError: 'AudioDecoder' object has no attribute 'path'

## Download Augmentation Datasets
We will now download the Room Impulse Responses (RIR) and background noise datasets (AudioSet and FMA). These are crucial for making the wake word model robust to real-world environments.

In [60]:
# Re-running the download and conversion with the fix for the AudioDecoder subscripting error
import datasets
import scipy.io.wavfile
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 1. Download MIT RIR data
rir_dir = './mit_rirs'
os.makedirs(rir_dir, exist_ok=True)
print('Downloading MIT Environmental Impulse Responses...')
# Loading dataset; we use standard loading and ensure we handle the audio column correctly
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')

for row in tqdm(rir_dataset, desc='Saving RIRs'):
    audio_data = row['audio']
    name = os.path.basename(audio_data['path'])
    audio_array = audio_data['array']
    scipy.io.wavfile.write(os.path.join(rir_dir, name), 16000, (audio_array*32767).astype(np.int16))

# 2. Download AudioSet (subset)
audioset_dir = './audioset_16k'
os.makedirs('audioset', exist_ok=True)
os.makedirs(audioset_dir, exist_ok=True)
if not os.path.exists('audioset/bal_train09.tar'):
    print('Downloading AudioSet tarball...')
    !wget -O audioset/bal_train09.tar https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/bal_train09.tar
!cd audioset && tar -xf bal_train09.tar

audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc='Converting AudioSet to 16k wav'):
    audio_data = row['audio']
    name = os.path.basename(audio_data['path']).replace('.flac', '.wav')
    scipy.io.wavfile.write(os.path.join(audioset_dir, name), 16000, (audio_data['array']*32767).astype(np.int16))

# 3. Download FMA (small subset)
fma_dir = './fma_16k'
os.makedirs('fma', exist_ok=True)
os.makedirs(fma_dir, exist_ok=True)
if not os.path.exists('fma/fma_xs.zip'):
    print('Downloading FMA zip...')
    !wget -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
!cd fma && unzip -qo fma_xs.zip

fma_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('fma/fma_small').glob('**/*.mp3')]})
fma_dataset = fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
for row in tqdm(fma_dataset, desc='Converting FMA to 16k wav'):
    audio_data = row['audio']
    name = os.path.basename(audio_data['path']).replace('.mp3', '.wav')
    scipy.io.wavfile.write(os.path.join(fma_dir, name), 16000, (audio_data['array']*32767).astype(np.int16))

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs:   0%|          | 0/270 [00:00<?, ?it/s]


TypeError: 'torchcodec.decoders.AudioDecoder' object is not subscriptable

In [88]:
import datasets
import scipy.io.wavfile
import os
import numpy as np
from tqdm import tqdm

# 1. Download MIT RIR data
rir_dir = './mit_rirs'
os.makedirs(rir_dir, exist_ok=True)
print('Downloading MIT Environmental Impulse Responses...')

# Loading dataset without streaming to ensure full access
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train')

for i, row in enumerate(tqdm(rir_dataset, desc='Saving RIRs')):
    audio_data = row['audio']

    # Handle AudioDecoder or dictionary formats
    if isinstance(audio_data, dict):
        array = audio_data['array']
        path = audio_data.get('path', f'rir_{i}.wav')
    else:
        # For objects like AudioDecoder, we access attributes
        array = np.array(audio_data.array if hasattr(audio_data, 'array') else audio_data['array'])
        path = getattr(audio_data, 'path', f'rir_{i}.wav')

    name = os.path.basename(path)
    if not name.endswith('.wav'):
        name += '.wav'

    # Convert to 16-bit PCM
    scipy.io.wavfile.write(os.path.join(rir_dir, name), 16000, (array * 32767).astype(np.int16))

print(f'Verification: Found {len(os.listdir(rir_dir))} files in {rir_dir}')

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs: 100%|██████████| 270/270 [00:00<00:00, 708.75it/s]

Verification: Found 270 files in ./mit_rirs


In [79]:
import datasets
import scipy.io.wavfile
import os
import numpy as np
from tqdm import tqdm

# 2. Download AudioSet (subset) via streaming
audioset_dir = './audioset_16k'
os.makedirs(audioset_dir, exist_ok=True)

print('Streaming AudioSet subset...')
try:
    # Correcting config name to 'balanced' as per the dataset's available configs
    audioset_dataset = datasets.load_dataset('agkphysics/AudioSet', 'balanced', split='train', streaming=True)

    # We will save 500 samples for robust augmentation
    max_samples = 500
    for i, row in enumerate(tqdm(audioset_dataset, total=max_samples, desc='Converting AudioSet')):
        if i >= max_samples:
            break

        audio_data = row['audio']
        audio_array = audio_data['array']

        # Standardize to 16kHz mono .wav
        name = f'audioset_{i}.wav'
        scipy.io.wavfile.write(os.path.join(audioset_dir, name), 16000, (audio_array * 32767).astype(np.int16))
    print(f'Successfully saved {max_samples} AudioSet samples.')
except Exception as e:
    print(f'Error streaming AudioSet: {e}')

Streaming AudioSet subset...


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Converting AudioSet: 100%|██████████| 500/500 [00:18<00:00, 27.54it/s]

Successfully saved 500 AudioSet samples.


In [76]:
import datasets
import scipy.io.wavfile
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 3. Download FMA (small subset)
fma_dir = './fma_16k'
os.makedirs('fma', exist_ok=True)
os.makedirs(fma_dir, exist_ok=True)

if not os.path.exists('fma/fma_xs.zip'):
    print('Downloading FMA zip...')
    !wget -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip

print('Unzipping FMA...')
!cd fma && unzip -qo fma_xs.zip

fma_files = [str(i) for i in Path('fma/fma_small').glob('**/*.mp3')]
fma_dataset = datasets.Dataset.from_dict({'audio': fma_files})
fma_dataset = fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))

for i, row in enumerate(tqdm(fma_dataset, desc='Converting FMA')):
    # Standard decoding via dict access
    audio_array = row['audio']['array']
    name = f'fma_{i}.wav'
    scipy.io.wavfile.write(os.path.join(fma_dir, name), 16000, (audio_array * 32767).astype(np.int16))

Unzipping FMA...


Converting FMA: 100%|██████████| 210/210 [00:14<00:00, 14.20it/s]


In [61]:
import os

# Verification script to ensure the wav files exist before moving to training
for directory in ['audioset_16k', 'fma_16k', 'mit_rirs']:
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.wav')]
        print(f"Directory '{directory}': {len(files)} files found.")
        if files:
            print(f"Sample files: {files[:3]}")
    else:
        print(f"Directory '{directory}' does not exist.")

Directory 'audioset_16k': 0 files found.
Directory 'fma_16k': 0 files found.
Directory 'mit_rirs': 0 files found.


In [42]:
import os
import random
from IPython.display import Audio, display

samples_dir = 'generated_samples'
if os.path.exists(samples_dir):
    sample_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]

    if sample_files:
        random_sample = random.choice(sample_files)
        sample_path = os.path.join(samples_dir, random_sample)
        print(f'Playing random sample: {sample_path}')
        display(Audio(sample_path, autoplay=True))
    else:
        print('No .wav files found in the directory. The generation might still be running or failed.')
else:
    print(f'Directory {samples_dir} does not exist.')

Playing random sample: generated_samples/209.wav


In [36]:
import os
import random
from IPython.display import Audio, display

samples_dir = 'generated_samples'
sample_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]

if sample_files:
    random_sample = random.choice(sample_files)
    sample_path = os.path.join(samples_dir, random_sample)
    print(f'Playing another random sample: {sample_path}')
    display(Audio(sample_path, autoplay=True))
else:
    print('No wav samples found.')

Playing another random sample: generated_samples/315.wav


In [34]:
import os
import random
from IPython.display import Audio, display

samples_dir = 'generated_samples'
sample_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]

if sample_files:
    random_sample = random.choice(sample_files)
    sample_path = os.path.join(samples_dir, random_sample)
    print(f'Playing another random sample: {sample_path}')
    display(Audio(sample_path, autoplay=True))
else:
    print('No wav samples found. Ensure the generation cell finished.')

Playing another random sample: generated_samples/228.wav


In [30]:
import os
import random
from IPython.display import Audio, display

samples_dir = 'generated_samples'
sample_files = [f for f in os.listdir(samples_dir) if f.endswith('.wav')]

if sample_files:
    random_sample = random.choice(sample_files)
    sample_path = os.path.join(samples_dir, random_sample)
    print(f'Playing random sample with spelling "hey koh dah": {sample_path}')
    display(Audio(sample_path, autoplay=True))
else:
    print('No samples found. Check the generation logs above.')

Playing random sample with spelling "hey koh dah": generated_samples/903.wav


## Download Augmentation Datasets
We will now download the Room Impulse Responses (RIR) and background noise datasets (AudioSet and FMA). These are crucial for making the wake word model robust to real-world environments.

In [52]:
# This cell downloads and processes the datasets. It may take some time.
import datasets
import scipy.io.wavfile
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

# 1. Download MIT RIR data
rir_dir = './mit_rirs'
if not os.path.exists(rir_dir):
    os.makedirs(rir_dir, exist_ok=True)
    print('Downloading MIT Environmental Impulse Responses...')
    rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
    for row in tqdm(rir_dataset, desc='Saving RIRs'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(rir_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 2. Download AudioSet (subset)
audioset_dir = './audioset_16k'
if not os.path.exists(audioset_dir):
    os.makedirs('audioset', exist_ok=True)
    os.makedirs(audioset_dir, exist_ok=True)
    print('Downloading AudioSet subset...')
    !wget -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar

    audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
    audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset, desc='Converting AudioSet to 16k wav'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(os.path.join(audioset_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 3. Download FMA (small subset)
fma_dir = './fma_16k'
if not os.path.exists(fma_dir):
    os.makedirs('fma', exist_ok=True)
    os.makedirs(fma_dir, exist_ok=True)
    print('Downloading FMA subset...')
    !wget -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
    !cd fma && unzip -q fma_xs.zip

    fma_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('fma/fma_small').glob('**/*.mp3')]})
    fma_dataset = fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset, desc='Converting FMA to 16k wav'):
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(os.path.join(fma_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/936 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs: 0it [00:05, ?it/s]


TypeError: 'torchcodec.decoders.AudioDecoder' object is not subscriptable

In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [121]:
# Downloads pre-generated spectrogram features for various negative datasets.
import os

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname
        zip_path = os.path.join(output_dir, fname)
        print(f"Downloading {fname}...")
        !wget -q -O {zip_path} {link}
        print(f"Unzipping {fname}...")
        !unzip -q {zip_path} -d {output_dir}
        os.remove(zip_path)

print("Negative datasets are ready.")

Unzipping dinner_party.zip...
Unzipping dinner_party_eval.zip...
Unzipping no_speech.zip...
Unzipping speech.zip...
Negative datasets are ready.


In [117]:
import yaml
import os

config = {}
config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hey_koda"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    }
]

config["training_steps"] = [10000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 128
config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500
config["maximization_metric"] = "average_viable_recall"

# Fix for KeyError: Explicitly define minimization metrics
config["minimization_metric"] = None
config["target_minimization"] = 0.9

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

print("Training configuration updated and saved to training_parameters.yaml")

Training configuration updated and saved to training_parameters.yaml


In [ ]:
import shutil
import os

# Ensure the model directory is fresh for the new run with complete data
model_dir = 'trained_models/hey_koda'
if os.path.exists(model_dir):
    shutil.rmtree(model_dir)
    print(f"Cleaned up {model_dir} for a fresh start with negative datasets.")

# Restart the training process
!python3 -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 0 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

Cleaned up trained_models/hey_koda for a fresh start with negative datasets.
INFO:absl:Loading and analyzing data sets.
2026-04-30 11:53:40.528557: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777550020.530025   26932 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────

In [120]:
import os

# Verify negative dataset directories
base_neg_path = 'negative_datasets'
subdirs = ['speech', 'no_speech']

print("--- Negative Dataset Verification ---")
if not os.path.exists(base_neg_path):
    print(f"[!] CRITICAL: Base directory '{base_neg_path}' does not exist.")
else:
    for sub in subdirs:
        path = os.path.join(base_neg_path, sub)
        if os.path.exists(path):
            # Look for .mmap or .npy or other feature files
            files = [f for f in os.listdir(path) if not f.startswith('.')]
            print(f"Directory '{path}': {len(files)} items found.")
            if len(files) > 0:
                print(f"  Samples: {files[:3]}")
            else:
                print(f"  [!] WARNING: '{path}' is empty. This will affect model robustness.")
        else:
            print(f"[!] WARNING: Subdirectory '{path}' does not exist.")

--- Negative Dataset Verification ---
[!] CRITICAL: Base directory 'negative_datasets' does not exist.


In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from google.colab import files

files.download(f"trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")